In [1]:
pip install transformers langchain pandas tqdm


Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import pandas as pd
from transformers import pipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm

# === Konfigurasjon ===
MAPPE = "/home/jovyan/digdir-camp-2025-desKI/finetuning/rag_and_lora/data_txt/_docs/ansattporten" 
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
CSV_FIL = "qa_mistral.csv"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 50
MAX_TOKENS = 150

# === Laster pipeline med god instruksjonsmodell ===
generator = pipeline(
    "text-generation",
    model=MODEL_ID,
    device_map="auto",  # Bruk begge GPUene
    torch_dtype="auto"
)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda:0


In [7]:
# === Finn .txt-filer rekursivt ===
def finn_txt_filer(rotmappe):
    for root, _, filer in os.walk(rotmappe):
        for fil in filer:
            if fil.endswith(".txt"):
                yield os.path.join(root, fil)

# === Split tekst i mindre biter (for kontrollert generering) ===
def chunk_tekst(tekst):
    splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    return splitter.split_text(tekst)

print(f"{len(chunks)} chunks i {filsti}")


# === Prompt som er tilpasset factual QA ===
def generer_qa(chunk):
    prompt = f"""Du er en hjelpsom assistent. Lag ett spørsmål og svar basert på teksten under. Svarene skal være korte og konkrete.

Tekst:
\"\"\"
{chunk}
\"\"\"

Spørsmål og svar:
"""
    
    try:
        out = generator(prompt, max_new_tokens=MAX_TOKENS, do_sample=False, temperature=0.7)[0]["generated_text"]
        return out[len(prompt):].strip()
    except Exception as e:
        print(f"⚠️ Feil: {e}")
        return ""

# === Ekstraher spm/svar fra modellens utputt ===
def parse_qa(generert_tekst):
    qas = []
    linjer = [l.strip() for l in generert_tekst.split("\n") if l.strip()]
    for i in range(0, len(linjer)-1, 2):
        spm = linjer[i].lstrip("1234567890.- ").strip()
        svar = linjer[i+1].lstrip("-→ ").strip()
        if spm and svar:
            qas.append({"spm": spm, "svar": svar})
    return qas

# === Kjør prosessen ===
alle_qas = []

for filsti in tqdm(list(finn_txt_filer(MAPPE))):
    with open(filsti, "r", encoding="utf-8") as f:
        tekst = f.read()

    chunks = chunk_tekst(tekst)
    for chunk in chunks:
        generert = generer_qa(chunk)
        qas = parse_qa(generert)
        alle_qas.extend(qas)

# === Lagre resultat ===
df = pd.DataFrame(alle_qas)
df.to_csv(CSV_FIL, index=False, quoting=1)
print(f"\n✅ Ferdig! {len(df)} QA-par lagret i: {CSV_FIL}")

85 chunks i /home/jovyan/digdir-camp-2025-desKI/finetuning/rag_and_lora/data_txt/_docs/ansattporten/ansattporten_allmennsky.txt


  0%|          | 0/10 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
 10%|█         | 1/10 [00:05<00:50,  5.61s/it]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSIT


✅ Ferdig! 239 QA-par lagret i: qa_mistral.csv
